In [ ]:
import tabula
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

In [ ]:
folders = {"tous":"../data/tous",
          "femmes":"../data/femmes",
          "hommes":"../data/hommes"}

combined_df_list = []
for key in folders:
    dfs = []
    folder = folders[key]
    pdf_paths = os.listdir(folder)
    for path in pdf_paths:
        # Extrait tous les tableaux du PDF
        file_path = os.path.join(folder, path)
        tables = tabula.read_pdf(file_path, pages="all", multiple_tables=True)
        for table in tables:
            df = table.copy()
            df.rename(columns={df.columns[0]: "Localisation", df.columns[1]:"Effectif"}, inplace=True)
            if "Taux Taux" in df.columns:
                df[["Taux standardisés (Europe)", "Taux standardisés (Monde)", "pou","belle"]] = df["Taux Taux"].str.split(" ", expand=True)
                df.drop(columns=["Taux Taux","pou","belle"], inplace=True)
            df.drop(index=[0,1,2,3], inplace=True)
            df["Année"] = path.split(" ")[0]
            df["Année"] = df["Année"].astype(int)
            df["Effectif"] = df["Effectif"].str.replace(" ", "").astype(int)
            df["Taux"] = df["Taux"].str.replace(",", ".").astype(float)
            df["Taux standardisés (Europe)"] = df["Taux standardisés (Europe)"].str.replace(",", ".").astype(float)
            df["Taux standardisés (Monde)"] = df["Taux standardisés (Monde)"].str.replace(",", ".").astype(float)
            dfs.append(df)

    # Combine tous les DataFrames en un seul
    combined_df = pd.concat(dfs, ignore_index=True)
    combined_df["Sexe"] = key
    combined_df.to_excel(f"../data/table_{key}.xlsx", index=False)
    combined_df_list.append(combined_df)
df = pd.concat(combined_df_list, ignore_index=True)
df.to_excel("../data/combined_tables.xlsx", index=False)

In [ ]:
df["Localisation"].replace("/","-", regex=True, inplace=True)
df["Localisation"].replace("di us","diffus", regex=True, inplace=True)
df["Localisation"].unique()

In [ ]:
test=False

for n,key in enumerate(folders):
    if n>0 and test:
        break
    print(key)
    filtre1 = df["Sexe"].str.contains(key, na=False)
    # display(df.loc[filtre1])
    print(df.loc[filtre1]["Localisation"].nunique())
    for m,loc in enumerate(df.loc[filtre1]["Localisation"].unique()):
        if m>0 and test:
            break
        filtre2 = df["Localisation"].str.contains(loc, na=False)
        df_plot = df.loc[filtre1 * filtre2]

        for c in ["Effectif", "Taux", "Taux standardisés (Europe)", "Taux standardisés (Monde)"]:
            print(f"Graphique pour {loc} - {c} - {key}")

            try:
                # Régression linéaire
                x = df_plot["Année"]
                y = df_plot[c]
                a, b = np.polyfit(x, y, 1)
                x_fit = np.linspace(x.min(), x.max(), 100)
                y_fit = a * x_fit + b

                # R²
                y_pred = a * x + b
                r2 = r2_score(y, y_pred)
            except Exception as e:
                print(f"Erreur lors du calcul de la régression linéaire pour {loc} - {c} - {key}: {e}")
                continue

            # Affichage du graphique
            plt.figure(figsize=(10, 6))  # Taille de la figure
            plt.plot(df_plot["Année"], df_plot[c],
                    marker='o',  # Ajoute des marqueurs pour chaque point
                    color='blue',  # Couleur de la ligne
                    label=c + " " + loc + " " + key)  # Label pour la légende

            plt.plot(x_fit, y_fit, linestyle='-', color='red', label=f"Régression linéaire: y = {a:.2f}x + {b:.2f}")
            # Personnalisation du graphique
            plt.title(f"Évol {c}, cancer {loc} par an, {key}", fontsize=14)
            plt.xlabel("Année", fontsize=12)
            plt.ylabel(c + " " + loc, fontsize=12)
            # Définition des limites de l'axe Y
            plt.ylim(ymin=df_plot[c].min()*0.8, ymax=df_plot[c].max()*1.2)  # Ajuste ces valeurs selon tes besoins
            plt.grid(True, linestyle='--', alpha=0.7)  # Grille discrète
            plt.xticks(df_plot["Année"])  # Affiche toutes les années sur l'axe X
            plt.legend()

            # Affiche le graphique
            os.makedirs(f"../data/graphes/{c}/{key}", exist_ok=True)
            plt.savefig(f"../data/graphes/{c}/{key}/{key}_{loc}_{c}.svg", format="svg", bbox_inches="tight")
            plt.show()
            plt.close()
            



In [ ]:
df_plot2=df.copy()

In [ ]:
pest_list = ["Lymphome diffus à grandes cellules B", "Prostate", "Poumon, bronche et trachée", "Sein", "Myélome multiple et plasmocytome", "LLC - Lymphome lymphocytique", "Leucémie aiguë myéloïde", "Syndrome myélodysplasique"]
len(pest_list)

In [ ]:
def is_in_pest(localisation):
    for pest in pest_list:
        if pest in localisation:
            return True
    return False

In [ ]:
df_plot2["lien pesticide"] = df_plot2["Localisation"].apply(is_in_pest)


In [ ]:
df_plot3 = df_plot2[df_plot2["lien pesticide"]==False].copy()
df_plot3 =df_plot3[df_plot3["Localisation"]!="TOTAL"]
df_plot3 =df_plot3[df_plot3["Sexe"]=="tous"].copy()

In [ ]:

df_plot2 = df_plot2[df_plot2["lien pesticide"]==True].copy()
df_plot2 = df_plot2[df_plot2["Sexe"]=="tous"].copy()


In [ ]:
variable = "Effectif"
# variable = "Taux standardisés (Europe)"

In [ ]:
dddf2 = df_plot2.groupby("Année")[[variable]].sum().copy()
dddf2

In [ ]:
# dddf3 =df_plot3.groupby("Année")[["Effectif"]].sum().copy()
dddf3 =df_plot3.groupby("Année")[[variable]].sum().copy()
dddf3

In [ ]:
dddf = df[df["Localisation"]=="TOTAL"].copy()
dddf = dddf[dddf["Sexe"]=="tous"].copy()
dddf

In [ ]:
for n,i in enumerate(dddf[variable]):
    print(i)
    print(dddf2.iloc[n].values[0])
    print(dddf3.iloc[n].values[0])
    print(i - dddf2.iloc[n].values[0] - dddf3.iloc[n].values[0])
    print("-----")

In [ ]:
c= variable
key="tous"
try:
    # Régression linéaire
    x = dddf2.index
    y = dddf2[c]
    a, b = np.polyfit(x, y, 1)
    x_fit = np.linspace(x.min(), x.max(), 100)
    y_fit = a * x_fit + b

    # R²
    y_pred = a * x + b
    r2 = r2_score(y, y_pred)
except Exception as e:
    print(e)

# Affichage du graphique
plt.figure(figsize=(10, 6))  # Taille de la figure
plt.plot(dddf2.index, dddf2[c],
        marker='o',  # Ajoute des marqueurs pour chaque point
        color='blue',  # Couleur de la ligne
        label="liés aux pesticides")  # Label pour la légende



plt.plot(x_fit, y_fit, linestyle='-', color='red', label=f"Régression linéaire: y = {a:.2f}x + {b:.2f}")


try:
    # Régression linéaire
    x = dddf3.index
    y = dddf3[c]
    a, b = np.polyfit(x, y, 1)
    x_fit = np.linspace(x.min(), x.max(), 100)
    y_fit = a * x_fit + b

    # R²
    y_pred = a * x + b
    r2 = r2_score(y, y_pred)
except Exception as e:
    print(e)

# Affichage du graphique

plt.plot(dddf3.index, dddf3[c],
        marker='o',  # Ajoute des marqueurs pour chaque point
        color='green',  # Couleur de la ligne
        label="non liés aux pesticides")  # Label pour la légende



plt.plot(x_fit, y_fit, linestyle='-', color='red', label=f"Régression linéaire: y = {a:.2f}x + {b:.2f}")



# Personnalisation du graphique
plt.title(f"Évol {c}, cancers par an, {key}", fontsize=14)
plt.xlabel("Année", fontsize=12)
plt.ylabel(c + " " + loc, fontsize=12)
# Définition des limites de l'axe Y
plt.ylim(ymin=min([dddf2[c].min(),dddf3[c].min()])*0.8, ymax=max(dddf2[c].max(),dddf2[c].max())*1.2)  # Ajuste ces valeurs selon tes besoins
plt.grid(True, linestyle='--', alpha=0.7)  # Grille discrète
plt.xticks(dddf2.index)  # Affiche toutes les années sur l'axe X
plt.legend()

plt.savefig(f"../data/graphes/evol_{c}.svg", format="svg", bbox_inches="tight")
plt.show()
